# Dataverse Exploration

First contact with the RNM-Dev environment data via Python.

**Goals of this notebook:**
1. Confirm authentication works
2. List tables that have data
3. Count records per relevant table for churn modeling
4. Explore the schema of the `account` table
5. Export a raw sample to `data/raw/` for the EDA notebook (Block 3)


## Setup

In [1]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from src.dataverse_client import DataverseClient

client = DataverseClient()
print(f"Client initialized for: {client.config.dataverse_url}")


Client initialized for: https://rnmdev.api.crm2.dynamics.com


## 1. List all tables in the environment

In [2]:
tables = client.list_tables()
tables_df = pd.DataFrame(tables)
print(f"Total tables in environment: {len(tables_df)}")
tables_df.head()


Total tables in environment: 925


,MetadataId,IsCustomEntity,LogicalName,EntitySetName,DisplayName
0,5575cfd2-5bae-4a26-921e-d224515408ec,True,aaduser,aadusers,{'LocalizedLabels': [{'Label': 'Microsoft Entr...
1,70816501-edb9-4740-a16c-6a5efbc05d84,False,account,accounts,"{'LocalizedLabels': [{'Label': 'Account', 'Lan..."
2,7634d589-4baa-471b-add7-892f27108731,False,aciviewmapper,aciviewmappers,{'LocalizedLabels': [{'Label': 'ACIViewMapper'...
3,03d2c2d7-f19e-4e42-a207-861d601763c1,False,actioncard,actioncards,"{'LocalizedLabels': [{'Label': 'Action Card', ..."
4,08e364fb-c780-45c5-8db8-df0271d12f5b,False,actioncardusersettings,actioncardusersettingsset,{'LocalizedLabels': [{'Label': 'Action Card Us...


### Filter to standard CRM tables we care about for churn prediction

For account churn modeling, the relevant tables are:
- `account` — the entity we're predicting churn for
- `contact` — people associated with accounts (decision makers)
- `opportunity` — sales pipeline (won/lost is a HUGE churn signal)
- `incident` — support cases (frustrated customers churn)
- `activitypointer` — emails/calls/meetings (engagement signal)


In [3]:
key_tables = ["account", "contact", "opportunity", "incident", "activitypointer"]
key_tables_df = tables_df[tables_df["LogicalName"].isin(key_tables)][
    ["LogicalName", "EntitySetName", "IsCustomEntity"]
].reset_index(drop=True)
key_tables_df


,LogicalName,EntitySetName,IsCustomEntity
0,account,accounts,False
1,activitypointer,activitypointers,False
2,contact,contacts,False


## 2. Count records per key table

In [4]:
counts = []
for _, row in key_tables_df.iterrows():
    entity_set = row["EntitySetName"]
    logical = row["LogicalName"]
    n = client.count_records(entity_set)
    counts.append({"table": logical, "entity_set": entity_set, "count": n})

counts_df = pd.DataFrame(counts).sort_values("count", ascending=False).reset_index(drop=True)
counts_df


,table,entity_set,count
0,activitypointer,activitypointers,78
1,contact,contacts,13
2,account,accounts,10


## 3. Explore the `account` table

Let's pull a small set of records to understand the structure.


In [5]:
# Select a focused set of fields relevant to churn signaling
account_fields = [
    "accountid",
    "name",
    "revenue",
    "numberofemployees",
    "industrycode",
    "customertypecode",
    "address1_city",
    "address1_country",
    "createdon",
    "modifiedon",
    "statecode",
    "statuscode",
]

accounts_df = client.to_dataframe("accounts", select=account_fields, top=10)
print(f"Shape: {accounts_df.shape}")
accounts_df


Shape: (10, 15)


,@odata.etag,customertypecode,accountid,_transactioncurrencyid_value,address1_city,numberofemployees,address1_composite,address1_country,modifiedon,statuscode,name,industrycode,statecode,revenue,createdon
0,"W/""2222156""",None,8a4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Renton,9500,Renton\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Fourth Coffee (sample),None,0,100000.0,2026-01-29T17:31:40Z
1,"W/""2222158""",None,8c4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Dallas,6000,Dallas\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,"Litware, Inc. (sample)",None,0,20000.0,2026-01-29T17:31:42Z
2,"W/""2222161""",None,8e4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Santa Cruz,4300,Santa Cruz\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Adventure Works (sample),None,0,60000.0,2026-01-29T17:31:42Z
3,"W/""2222163""",None,904ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Lynnwood,2700,Lynnwood\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,"Fabrikam, Inc. (sample)",None,0,80000.0,2026-01-29T17:31:42Z
4,"W/""2222164""",None,924ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Los Angeles,2900,Los Angeles\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Blue Yonder Airlines (sample),None,0,10000.0,2026-01-29T17:31:42Z
5,"W/""2222165""",None,944ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Redmond,2900,Redmond\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,City Power & Light (sample),None,0,100000.0,2026-01-29T17:31:42Z
6,"W/""2222166""",None,964ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Redmond,1500,Redmond\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Contoso Pharmaceuticals (sample),None,0,60000.0,2026-01-29T17:31:42Z
7,"W/""2222167""",None,984ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Missoula,4800,Missoula\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Alpine Ski House (sample),None,0,90000.0,2026-01-29T17:31:42Z
8,"W/""2222168""",None,9a4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Redmond,6200,Redmond\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,A. Datum Corporation (sample),None,0,10000.0,2026-01-29T17:31:42Z
9,"W/""2222169""",None,9c4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Phoenix,3900,Phoenix\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Coho Winery (sample),None,0,10000.0,2026-01-29T17:31:42Z


### What do those numeric codes mean?

`industrycode`, `customertypecode`, `statecode`, `statuscode` are **option sets** in Dataverse — they store integers internally but have human-readable labels in the UI. For example, `industrycode=1` means "Accounting", `statecode=0` means "Active".

In Block 3 (EDA), we'll bring in the metadata to translate these. For now, we work with raw codes.


In [6]:
# Get one full record with all fields, to see everything available
full_account = client.get_records("accounts", top=1)[0]
print(f"Total fields per account record: {len(full_account)}")
print("\nAll field names (first 30):")
for field in list(full_account.keys())[:30]:
    print(f"  {field}")


Total fields per account record: 151

All field names (first 30):
  @odata.etag
  telephone3
  address1_shippingmethodcode
  sharesoutstanding
  ownershipcode
  address1_freighttermscode
  _ownerid_value
  address1_upszone
  merged
  websiteurl
  address2_city
  _slainvokedid_value
  address1_postofficebox
  importsequencenumber
  preferredappointmentdaycode
  customertypecode
  utcconversiontimezonecode
  overriddencreatedon
  aging90
  stageid
  address1_latitude
  address1_utcoffset
  adx_createdbyipaddress
  _masterid_value
  lastonholdtime
  address2_fax
  accountid
  _transactioncurrencyid_value
  accountnumber
  participatesinworkflow


## 4. Export raw sample for the EDA notebook

In [7]:
# Pull all accounts with the focused fields
all_accounts = client.to_dataframe("accounts", select=account_fields)
print(f"Total accounts exported: {len(all_accounts)}")

# Save to data/raw/ (gitignored)
output_path = project_root / "data" / "raw" / "accounts_raw.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
all_accounts.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
all_accounts.head()


Total accounts exported: 10
Saved to: /Users/rafaelnedermaiorino/projects/churn-d365/data/raw/accounts_raw.csv


,@odata.etag,customertypecode,accountid,_transactioncurrencyid_value,address1_city,numberofemployees,address1_composite,address1_country,modifiedon,statuscode,name,industrycode,statecode,revenue,createdon
0,"W/""2222156""",None,8a4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Renton,9500,Renton\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Fourth Coffee (sample),None,0,100000.0,2026-01-29T17:31:40Z
1,"W/""2222158""",None,8c4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Dallas,6000,Dallas\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,"Litware, Inc. (sample)",None,0,20000.0,2026-01-29T17:31:42Z
2,"W/""2222161""",None,8e4ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Santa Cruz,4300,Santa Cruz\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Adventure Works (sample),None,0,60000.0,2026-01-29T17:31:42Z
3,"W/""2222163""",None,904ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Lynnwood,2700,Lynnwood\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,"Fabrikam, Inc. (sample)",None,0,80000.0,2026-01-29T17:31:42Z
4,"W/""2222164""",None,924ab64b-38fd-f011-8407-6045bd3b760b,9b9ed899-33fd-f011-8406-7ced8da86c93,Los Angeles,2900,Los Angeles\r\nU.S.,U.S.,2026-01-29T17:31:57Z,1,Blue Yonder Airlines (sample),None,0,10000.0,2026-01-29T17:31:42Z


## Next steps

- **Block 3** (EDA): full exploratory analysis on this exported CSV, plus joining with `contact`, `opportunity`, `incident`, `activitypointer` for feature engineering
- **Block 4**: baselines (Dummy, Logistic Regression) + MLP PyTorch trained on engineered features
- **Block 5**: refactor everything into the FastAPI service
